<a href="https://colab.research.google.com/github/annjgs/Fisica-Computacionalo/blob/Taller-5/Uniformidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile programa_uniformidad.cpp

using namespace std;

// Función de la prueba de uniformidad
void pruebaMomentos(const vector<double>& datos, int k) {
    int N = datos.size();
    double suma_potencias = 0.0;

    // <x^k> = (1/N) * sum(x_i^k)
    for (double x : datos) {
        suma_potencias += pow(x, k);
    }
    double momento_muestral = suma_potencias / N;

    // Valor esperado = 1 / (k + 1)
    double valor_teorico = 1.0 / (k + 1);

    // Desviación y Aleatoriedad (1/sqrt(N))
    double desviacion = abs(momento_muestral - valor_teorico);
    double umbral = 1.0 / sqrt(N);

    cout << fixed << setprecision(6);
    cout << "--- Prueba de Uniformidad (k=" << k << ") ---" << endl;
    cout << "Momento Muestral: " << momento_muestral << endl;
    cout << "Valor Teórico:    " << valor_teorico << endl;
    cout << "Desviación Real:  " << desviacion << endl;
    cout << "Umbral (1/sqrt(N)): " << umbral << endl;

    if (desviacion <= umbral) {
        cout << "ESTADO: PASA la prueba." << endl;
    } else {
        cout << "ESTADO: FALLA la prueba." << endl;
    }
}

int main() {
    int M = 9; // Cantidad de datos del problema 1
    double A = 10.0, B = 20.0;
    vector<double> xi_escalados;

    cout << "Punto 1 y 2: Generando y escalando datos a [" << A << ", " << B << "]" << endl;

    // Generación y escalado: x_i = A + (B - A) * r_i
    for (int i = 1; i <= M; ++i) {
        double ri = (double)i / (M + 1); // Normalizado [0, 1]
        double xi = A + (B - A) * ri;    // Escalado [10, 20]
        xi_escalados.push_back(xi);
        cout << "x" << i << ": " << xi << endl;
    }

    cout << "--- Validando con Prueba de Momentos ---" << endl;

    // volvemos al rango [0,1]
    vector<double> datos_para_prueba;
    for (double val : xi_escalados) {
        datos_para_prueba.push_back((val - A) / (B - A));
    }


    pruebaMomentos(datos_para_prueba, 1);

    return 0;
}

Writing programa_uniformidad.cpp


In [ ]:
!g++ mapeo_caotico.cpp -o mapeo
!./mapeo

cc1plus: fatal error: mapeo_caotico.cpp: No such file or directory
compilation terminated.
/bin/bash: line 1: ./mapeo: No such file or directory


In [ ]:
%%writefile generador_malo.cpp
#include <iostream>
#include <vector>
#include <cmath>
#include <iomanip>
#include <set>

using namespace std;

int main() {
    // variables del problema
    int a = 57, c = 1, M = 256;
    int r_actual = 10; // Semilla r1

    vector<int> secuencia;
    set<int> vistos;
    int periodo = 0;

    // Generar secuencia y determinar el período
    cout << "--- Generando Secuencia LCG ---" << endl;
    while (vistos.find(r_actual) == vistos.end()) {
        vistos.insert(r_actual);
        secuencia.push_back(r_actual);

        // Fórmula: r_{i+1} = (a * r_i + c) mod M
        r_actual = (a * r_actual + c) % M;
        periodo++;
    }

    cout << "Periodo detectado: " << periodo << " (Maximo posible: " << M << ")" << endl;
    if (periodo < M) cout << " El periodo es incompleto." << endl;

    // Normalizar la secuencia al rango [0, 1]
    vector<double> ri_normalizados;
    for (int val : secuencia) {
        ri_normalizados.push_back((double)val / M);
    }

    //  prueba de uniformidad
    // <x^k> = (1/N) * sum(x_i^k)
    double suma = 0;
    for (double x : ri_normalizados) suma += x;
    double momento_muestral = suma / periodo;
    double momento_teorico = 1.0 / (1.0 + 1.0); // 1/(k+1) = 0.5

    double desviacion = abs(momento_muestral - momento_teorico);
    double umbral = 1.0 / sqrt(periodo);

    cout << fixed << setprecision(6) << endl;
    cout << " Resultado Prueba de Momentos (k=1) " << endl;
    cout << "Momento Muestral: " << momento_muestral << endl;
    cout << "Momento Teorico:  " << momento_teorico << endl;
    cout << "Desviacion:       " << desviacion << endl;
    cout << "Umbral (1/sqrt(N)): " << umbral << endl;

    if (desviacion < umbral) {
        cout << "RESULTADO: PASA la prueba de uniformidad." << endl;
    } else {
        cout << "RESULTADO: NO es uniforme ." << endl;
    }

    return 0;
}

In [ ]:
!g++ mapeo_caotico.cpp -o mapeo
!./mapeo

In [ ]:
%%writefile mapeo_caotico.cpp
#include <iostream>
#include <vector>
#include <cmath>
#include <iomanip>
#include <map>

using namespace std;

int main() {
    int N = 10000;
    double r = 4.0;
    double x = 0.4;
    vector<double> datos;

    // números mediante mapeo caotico
    for (int i = 0; i < N; ++i) {
        x = r * x * (1 - x);
        datos.push_back(x);
    }

    // Calculos estadisticos
    double suma = 0, suma_sq = 0, suma_cub = 0;
    for (double v : datos) {
        suma += v;
        suma_sq += pow(v, 2);
        suma_cub += pow(v, 3);
    }

    double media = suma / N;
    double varianza = (suma_sq / N) - pow(media, 2);
    double desviacion = sqrt(varianza);

    // Asimetría
    double m3 = 0;
    for (double v : datos) m3 += pow(v - media, 3);
    m3 /= N;
    double asimetria = m3 / pow(desviacion, 3);


    // Resultados
    cout << fixed << setprecision(4);
    cout << " ANALISIS ESTADISTICO (MAPEO CAOTICO) " << endl;
    cout << "Media:     " << media << " (Esperado: 0.5000)" << endl;
    cout << "Varianza:  " << varianza << " (Esperado: 0.0833)" << endl;
    cout << "Asimetria: " << asimetria << " (Esperado: 0.0000)" << endl;


    return 0;
}

Writing mapeo_caotico.cpp


In [ ]:
!g++ mapeo_caotico.cpp -o mapeo
!./mapeo

=== ANALISIS ESTADISTICO (MAPEO CAOTICO) ===
Media:     0.4968 (Esperado: 0.5000)
Varianza:  0.1258 (Esperado: 0.0833)
Asimetria: 0.0150 (Esperado: 0.0000)

=== HISTOGRAMA DE FRECUENCIAS ===
[0.0000-0.1000]: ******************** (2094)
[0.1000-0.2000]: ********* (922)
[0.2000-0.3000]: ******* (726)
[0.3000-0.4000]: ****** (645)
[0.4000-0.5000]: ****** (648)
[0.5000-0.6000]: ****** (637)
[0.6000-0.7000]: ****** (680)
[0.7000-0.8000]: ******* (706)
[0.8000-0.9000]: ********* (904)
[0.9000-1.0000]: ******************** (2038)
